In [ ]:
import cv2
import json
import os

video_ID = '1TEST'

vid_path = f'/home/edisonz/maneuver_heuristics25/clips/Parking-clip{video_ID}.mp4'
output_file = 'regions.json'

# Global state
drawing = False
current_polygon = []
parking_areas = []
mode = None  # Only "parking"

def click_event(event, x, y, flags, param):
    global current_polygon, drawing, mode
    if event == cv2.EVENT_LBUTTONDOWN and drawing:
        if mode == "parking":
            current_polygon.append((x, y))

def draw_polygon(img, points, color):
    if len(points) > 0:
        for point in points:
            cv2.circle(img, point, 3, color, -1)
    if len(points) > 1:
        for i in range(len(points) - 1):
            cv2.line(img, points[i], points[i + 1], color, 2)
        if len(points) > 2:
            cv2.line(img, points[-1], points[0], color, 2)

def draw_regions(img):
    for poly in parking_areas:
        draw_polygon(img, poly, (0, 255, 0))

def load_existing_regions(file_path):
    global parking_areas
    if os.path.exists(file_path):
        with open(file_path, 'r') as f:
            data = json.load(f)
            parking_areas = [tuple(map(tuple, region)) for region in data.get("parking_areas", [])]
        print(f"📂 Loaded existing parking areas from {file_path}")

def save_regions(file_path):
    with open(file_path, 'w') as f:
        json.dump({
            'parking_areas': parking_areas,
        }, f, indent=2)
    print(f"✅ Parking zones saved to {file_path}")

def main(video_path, output_file='regions.json'):
    global drawing, current_polygon, parking_areas, mode

    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print("❌ Failed to open video.")
        return

    fps = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    frame_idx = 0  # Start at first frame

    load_existing_regions(output_file)

    cv2.namedWindow("Define Parking Zones")
    cv2.setMouseCallback("Define Parking Zones", click_event)

    while True:
        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
        ret, frame = cap.read()
        if not ret:
            print("❌ Failed to read frame.")
            break

        cropped_frame = frame[frame.shape[0] // 3:, :].copy()

        temp_img = cropped_frame.copy()
        cv2.putText(temp_img, f"Frame: {frame_idx}/{total_frames}", (10, 55),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 2)
        cv2.putText(temp_img, f"Mode: {mode or 'None'}", (10, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)
        cv2.putText(temp_img, "←/→: -/+1 sec | p: start | a: add | s: save | u: undo | c: cancel | r: reset | q: quit",
                    (10, temp_img.shape[0] - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 1)

        # Draw current polygon
        if drawing and mode == "parking":
            draw_polygon(temp_img, current_polygon, (0, 255, 0))

        # Draw saved polygons
        draw_regions(temp_img)

        cv2.imshow("Define Parking Zones", temp_img)
        key = cv2.waitKey(1) & 0xFF

        if key == ord('p'):
            mode = "parking"
            current_polygon = []
            drawing = True

        elif key == ord('r'):
            parking_areas = []
            current_polygon = []
            drawing = False
            mode = None
            print("🔄 All parking zones have been cleared.")

        elif key == ord('a'):
            if mode == "parking" and current_polygon:
                parking_areas.append(current_polygon[:])
                save_regions(output_file)
            current_polygon = []
            drawing = False
            mode = None

        elif key == ord('s'):
            save_regions(output_file)
            print("All regions have been saved.")

        elif key == ord('u'):
            if current_polygon:
                current_polygon.pop()

        elif key == ord('c'):
            current_polygon = []
            drawing = False
            mode = None

        elif key == 81:  # Left arrow
            frame_idx = max(0, frame_idx - int(fps))

        elif key == 83:  # Right arrow
            frame_idx = min(total_frames - 1, frame_idx + int(fps))

        elif key == ord('q'):
            print("👋 Exiting without saving...")
            break

    cap.release()
    cv2.destroyAllWindows()

if __name__ == "__main__":
    main(vid_path, output_file)


📂 Loaded existing parking areas from regionsB.json
👋 Exiting without saving...


In [ ]:
import cv2
import json
import os
import numpy as np

vid_path = f'/home/edisonz/maneuver_heuristics25/clips/Parking-clip{video_ID}.mp4'
region_file = '/home/edisonz/maneuver_heuristics25/regions.json'
line_output_file = 'lines.json'

current_line = []
all_lines = []
current_batch = []

parking_areas = []

def click_event(event, x, y, flags, param):
    global current_line, all_lines, current_batch
    if event == cv2.EVENT_LBUTTONDOWN:
        current_line.append((x, y))
        if len(current_line) == 2:
            all_lines.append(current_line[:])
            current_batch.append(current_line[:])
            current_line = []

def draw_polygon(img, points, color):
    if len(points) > 0:
        for point in points:
            cv2.circle(img, point, 3, color, -1)
    if len(points) > 1:
        for i in range(len(points) - 1):
            cv2.line(img, points[i], points[i + 1], color, 2)
        if len(points) > 2:
            cv2.line(img, points[-1], points[0], color, 2)

def draw_lines(img, lines, color=(255, 0, 0)):
    for line in lines:
        if len(line) == 2:
            cv2.line(img, line[0], line[1], color, 2)

def draw_regions(img):
    for poly in parking_areas:
        draw_polygon(img, poly, (0, 255, 0))

def load_existing_regions(file_path):
    global parking_areas
    if os.path.exists(file_path):
        with open(file_path, 'r') as f:
            data = json.load(f)
            parking_areas = [list(map(tuple, region)) for region in data.get("parking_areas", [])]
        print(f"📂 Loaded parking areas from {file_path}")

def load_existing_lines(file_path):
    global all_lines
    if os.path.exists(file_path):
        with open(file_path, 'r') as f:
            data = json.load(f)
            all_lines = [list(map(tuple, line)) for line in data.get("lines", [])]
        print(f"📂 Loaded existing lines from {file_path}")

def save_lines(file_path):
    with open(file_path, 'w') as f:
        json.dump({
            'lines': [list(map(list, line)) for line in all_lines],
        }, f, indent=2)
    print(f"✅ Parking lines saved to {file_path}")

# ----- main algorithm -----
def compute_vanishing_point(lines):
    A = []
    b = []
    for (x1, y1), (x2, y2) in lines:
        if x1 == x2:  # vertical line
            m = np.inf
            c = x1
        else:
            m = (y2 - y1) / (x2 - x1)
            c = y1 - m * x1

        if m == 0:
            continue

        if np.isinf(m):
            A.append([1, 0])
            b.append(c)
        else:
            A.append([-m, 1])
            b.append(c)

    A = np.array(A)
    b = np.array(b)
    vp, _, _, _ = np.linalg.lstsq(A, b, rcond=None)
    return vp

def fit_line_through_furthest_endpoints(lines, vp):
    vp = np.array(vp)
    furthest_points = []

    for (x1, y1), (x2, y2) in lines:
        p1 = np.array([x1, y1])
        p2 = np.array([x2, y2])
        d1 = np.linalg.norm(p1 - vp)
        d2 = np.linalg.norm(p2 - vp)
        if d1 > d2:
            furthest_points.append(p1)
        else:
            furthest_points.append(p2)

    furthest_points = np.array(furthest_points)
    m, c = np.polyfit(furthest_points[:, 0], furthest_points[:, 1], 1)
    return m, c, furthest_points

def extrapolate_points_left(vp, m, c, furthest_points, max_points=20, img_shape=None):
    """
    Extrapolate points to the LEFT of the leftmost point in furthest_points.
    """
    sorted_pts = furthest_points[np.argsort(furthest_points[:, 0])]

    i_vals = np.arange(len(sorted_pts))
    y_vals = np.clip(sorted_pts[:, 1], 1e-3, None)
    inv_y = 1.0 / y_vals

    A = np.vstack([i_vals, np.ones_like(i_vals)]).T
    b, a = np.linalg.lstsq(A, inv_y, rcond=None)[0]

    new_i = np.arange(-max_points, 0)  # only leftward

    extrapolated = []
    for i in new_i:
        new_inv_y_i = b * i + a
        if new_inv_y_i <= 0:
            continue

        new_y_i = 1.0 / new_inv_y_i
        if abs(m) < 1e-6:
            continue

        new_x_i = (new_y_i - c) / m

        if img_shape is not None:
            if not (0 <= new_x_i < img_shape[1] and 0 <= new_y_i < img_shape[0]):
                continue

        extrapolated.append((int(round(new_x_i)), int(round(new_y_i))))

    return extrapolated


def extrapolate_points_right(vp, m, c, furthest_points, max_points=20, img_shape=None):
    """
    Extrapolate points to the RIGHT of the rightmost point in furthest_points.
    """
    sorted_pts = furthest_points[np.argsort(furthest_points[:, 0])]

    i_vals = np.arange(len(sorted_pts))
    y_vals = np.clip(sorted_pts[:, 1], 1e-3, None)
    inv_y = 1.0 / y_vals

    A = np.vstack([i_vals, np.ones_like(i_vals)]).T
    b, a = np.linalg.lstsq(A, inv_y, rcond=None)[0]

    new_i = np.arange(len(sorted_pts), len(sorted_pts) + max_points)  # only rightward

    extrapolated = []
    for i in new_i:
        new_inv_y_i = b * i + a
        if new_inv_y_i <= 0:
            continue

        new_y_i = 1.0 / new_inv_y_i
        if abs(m) < 1e-6:
            continue

        new_x_i = (new_y_i - c) / m

        if img_shape is not None:
            if not (0 <= new_x_i < img_shape[1] and 0 <= new_y_i < img_shape[0]):
                continue

        extrapolated.append((int(round(new_x_i)), int(round(new_y_i))))

    return extrapolated

########################################################################################

def main(video_path, region_file='regions.json', line_output_file='lines.json'):
    global current_line, current_batch, all_lines, parking_areas

    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print("❌ Could not open video.")
        return

    fps = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    frame_idx = 0  # start at first frame

    load_existing_regions(region_file)
    load_existing_lines(line_output_file)

    cv2.namedWindow("Annotate Parking Lines")
    cv2.setMouseCallback("Annotate Parking Lines", click_event)

    while True:
        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
        ret, frame = cap.read()
        if not ret:
            print("❌ Failed to read frame.")
            break

        cropped_frame = frame[frame.shape[0] // 3:, :].copy()

        temp_img = cropped_frame.copy()
        cv2.putText(temp_img, f"Frame: {frame_idx}/{total_frames}", (10, 55),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 2)
        cv2.putText(temp_img, f"Lines: {len(all_lines)}", (10, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 2)
        cv2.putText(temp_img,
            "←/→: -/+1 sec | l: left extrap | r: right extrap | n: new batch | s: save | u: undo | c: clear | q: quit",
            (10, temp_img.shape[0] - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 1)

        draw_regions(temp_img)
        draw_lines(temp_img, all_lines)

        if current_line:
            cv2.circle(temp_img, current_line[0], 3, (255, 0, 0), -1)

        draw_lines(temp_img, all_lines)

        cv2.imshow("Annotate Parking Lines", temp_img)
        key = cv2.waitKey(1) & 0xFF

        if key == ord('n'):
            current_batch = []

        elif key == ord('s'):
            print("All lines have been saved.")
            save_lines(line_output_file)

        elif key == ord('u'):
            if current_line:
                current_line.pop()
            elif all_lines:
                all_lines.pop()

        elif key == ord('c'):
            all_lines = []
            current_line = []
            print("🔄 All lines have been cleared.")

        elif key == ord('l'):
            if not current_batch:
                print("⚠️ No existing lines to extrapolate from!")
                continue
            vp = compute_vanishing_point(all_lines)
            m, c, furthest_points = fit_line_through_furthest_endpoints(current_batch, vp)
            extrapolated_pts = extrapolate_points_left(vp, m, c, furthest_points, max_points=10, img_shape=cropped_frame.shape)
            for pt in extrapolated_pts:
                all_lines.append([pt, tuple(map(int, vp))])
            print(f"✨ Added {len(extrapolated_pts)} left extrapolated lines.")

        elif key == ord('r'):
            if not current_batch:
                print("⚠️ No existing lines to extrapolate from!")
                continue
            vp = compute_vanishing_point(all_lines)
            m, c, furthest_points = fit_line_through_furthest_endpoints(current_batch, vp)
            extrapolated_pts = extrapolate_points_right(vp, m, c, furthest_points, max_points=10, img_shape=cropped_frame.shape)
            for pt in extrapolated_pts:
                all_lines.append([pt, tuple(map(int, vp))])
            print(f"✨ Added {len(extrapolated_pts)} right extrapolated lines.")

        elif key == 81:  # Left arrow
            frame_idx = max(0, frame_idx - int(fps))  # back 1 sec

        elif key == 83:  # Right arrow
            frame_idx = min(total_frames - 1, frame_idx + int(fps))  # forward 1 sec

        elif key == ord('q'):
            print("👋 Exiting without saving...")
            break

    cap.release()
    cv2.destroyAllWindows()


if __name__ == "__main__":
    main(vid_path, region_file, line_output_file)


✨ Added 7 right extrapolated lines.
All lines have been saved.
✅ Parking lines saved to lines.json
👋 Exiting without saving...
